In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv('/kaggle/input/one-million-clicks-later/train.csv')
test = pd.read_csv('/kaggle/input/one-million-clicks-later/test.csv')
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train columns:", train.columns.tolist())
print(train.info())
print("Clicked Value Counts:")
print(train['clicked'].value_counts())


Train shape: (610310, 15)
Test shape: (152578, 14)
Train columns: ['user_id', 'video_id', 'video_duration', 'watch_time', 'liked', 'commented', 'subscribed_after', 'category', 'device', 'watch_time_of_day', 'recommended', 'clicked', 'timestamp', 'watch_percent', 'id']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610310 entries, 0 to 610309
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   user_id            610310 non-null  int64  
 1   video_id           610310 non-null  int64  
 2   video_duration     610310 non-null  float64
 3   watch_time         610310 non-null  float64
 4   liked              567185 non-null  object 
 5   commented          610310 non-null  int64  
 6   subscribed_after   610310 non-null  int64  
 7   category           610310 non-null  object 
 8   device             610310 non-null  object 
 9   watch_time_of_day  610310 non-null  object 
 10  recommended        610310 n

In [11]:
train['watch_ratio'] = train['watch_time'] / (train['video_duration'].replace(0, 1))
test['watch_ratio'] = test['watch_time'] / (test['video_duration'].replace(0, 1))
train['interaction_score'] = train['liked'] + (train['commented'] == 'yes').astype(int) + train['recommended']
test['interaction_score'] = test['liked'] + (test['commented'] == 'yes').astype(int) + test['recommended']
for period in ['Afternoon', 'Night', 'Morning', 'Evening']:
    train['is_' + period.lower()] = (train['watch_time_of_day'] == period).astype(int)
    test['is_' + period.lower()] = (test['watch_time_of_day'] == period).astype(int)

for col in ['video_duration', 'watch_time', 'watch_percent', 'watch_ratio']:
    train[f'log_{col}'] = np.log1p(train[col])
    test[f'log_{col}'] = np.log1p(test[col])

train['cat_dev'] = train['category'].astype(str) + '_' + train['device'].astype(str)
test['cat_dev'] = test['category'].astype(str) + '_' + test['device'].astype(str)
le = LabelEncoder()
le.fit(list(train['cat_dev']) + list(test['cat_dev']))
train['cat_dev'] = le.transform(train['cat_dev'])
test['cat_dev'] = le.transform(test['cat_dev'])


/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [12]:
label_cols = ['category', 'device', 'watch_time_of_day', 'cat_dev']
numeric_cols = ['video_id', 'video_duration', 'watch_time', 'liked', 'recommended', 'watch_percent',
                'watch_ratio', 'interaction_score', 'is_afternoon', 'is_night', 'is_morning', 'is_evening'] + \
                [f'log_{col}' for col in ['video_duration', 'watch_time', 'watch_percent', 'watch_ratio']]
for col in label_cols:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)
    le = LabelEncoder()
    le.fit(list(train[col]) + list(test[col]))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])
for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0)
    test[col] = pd.to_numeric(test[col], errors='coerce').fillna(0)
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)


In [13]:
features = label_cols + numeric_cols
X = train[features]
y = train['clicked']
test_X = test[features]

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, valid_idx in sss.split(X, y):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_val_probs = nb.predict_proba(X_valid)[:, 1]

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.01, 0.60, 0.01):
    f1 = f1_score(y_valid, nb_val_probs > thresh)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
print(f"Best Validation F1: {best_f1:.4f} at threshold {best_thresh:.2f}")
nb_test_probs = nb.predict_proba(test_X)[:, 1]
nb_test_pred = (nb_test_probs > best_thresh).astype(int)
submission_nb = pd.DataFrame({'id': test['id'], 'clicked': nb_test_pred})
submission_nb.to_csv('submission_nb.csv', index=False)
print(submission_nb.head(), submission_nb['clicked'].value_counts())



Best Validation F1: 0.2551 at threshold 0.02
       id  clicked
0   53363        1
1  293669        1
2   52195        1
3  260007        1
4  602213        1 clicked
1    152325
0       253
Name: count, dtype: int64
